### RECOMMENDATION QUALITY + HALLUCINATION

In [ ]:


from pathlib import Path
import sys
import os
import math
import numpy as np
import pandas as pd

from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS



PROJECT_ROOT = Path.cwd()

if not (
    PROJECT_ROOT / "Agents"
).exists():

    PROJECT_ROOT = (
        PROJECT_ROOT.parent
    )


print(
    "PROJECT_ROOT:",
    PROJECT_ROOT
)



if str(PROJECT_ROOT) not in sys.path:

    sys.path.append(
        str(PROJECT_ROOT)
    )


load_dotenv()



from Agents.query_agent import QueryAgent

from Agents.verifier_agent import (
    check_brand_match,
    check_product_type_match,
    check_feature_match,
    normalise_price
)


print(
    "Imports successful."
)

PROJECT_ROOT: c:\Users\srush\Desktop\Multi agent coordination
Imports successful.


C:\Users\srush\AppData\Local\Temp\ipykernel_13188\513366339.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:


RESULTS_FOLDER = (
    PROJECT_ROOT
    / "Results"
)

all_csv_files = list(
    RESULTS_FOLDER.glob(
        "*.csv"
    )
)

print("CSV files found:")

for file in all_csv_files:
    print(file.name)

CSV files found:
coordinated_pipeline_test_results.csv
multi_agent_without_coordination_results.csv
single_agent_test_results.csv


In [ ]:

def find_csv(
    include_terms,
    exclude_terms=None
):

    exclude_terms = (
        exclude_terms
        or []
    )

    matches = []

    for file in all_csv_files:

        name = (
            file.name
            .lower()
        )

        if all(
            term.lower() in name
            for term in include_terms
        ):

            if not any(
                term.lower() in name
                for term
                in exclude_terms
            ):
                matches.append(
                    file
                )

    return matches


single_candidates = find_csv(
    ["single"]
)

no_coord_candidates = (
    find_csv(
        ["coord"],
        exclude_terms=[
            "coordinated_pipeline"
        ]
    )
)

coord_candidates = find_csv(
    [
        "coordinated",
        "pipeline"
    ]
)


print(
    "\nSingle:",
    single_candidates
)

print(
    "\nNo coordination:",
    no_coord_candidates
)

print(
    "\nCoordinated:",
    coord_candidates
)


Single: [WindowsPath('c:/Users/srush/Desktop/Multi agent coordination/Results/single_agent_test_results.csv')]

No coordination: [WindowsPath('c:/Users/srush/Desktop/Multi agent coordination/Results/multi_agent_without_coordination_results.csv')]

Coordinated: [WindowsPath('c:/Users/srush/Desktop/Multi agent coordination/Results/coordinated_pipeline_test_results.csv')]


In [ ]:


SINGLE_PATH = (
    single_candidates[0]
)

NO_COORD_PATH = (
    no_coord_candidates[0]
)

COORD_PATH = (
    coord_candidates[0]
)


single_df = pd.read_csv(
    SINGLE_PATH
)

no_coord_df = pd.read_csv(
    NO_COORD_PATH
)

coordinated_df = pd.read_csv(
    COORD_PATH
)


print(
    "Single:",
    len(single_df)
)

print(
    "No coordination:",
    len(no_coord_df)
)

print(
    "Coordinated:",
    len(coordinated_df)
)

Single: 15
No coordination: 15
Coordinated: 15


In [ ]:

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


vector_db = FAISS.load_local(
    str(
        PROJECT_ROOT
        / "Data"
        / "Cleaned"
        / "faiss_index"
    ),
    embedding_model,
    allow_dangerous_deserialization=True
)


print(
    "FAISS loaded."
)

FAISS loaded.


In [ ]:


all_documents = list(
    vector_db.docstore
    ._dict
    .values()
)


asin_to_product = {}


for document in all_documents:

    asin = str(
        document.metadata.get(
            "parent_asin",
            ""
        )
    ).strip()

    if not asin:
        continue

    product = dict(
        document.metadata
    )

    product[
        "product_text"
    ] = document.page_content

    product[
        "description"
    ] = document.page_content

    asin_to_product[
        asin
    ] = product


print(
    "Products indexed:",
    len(asin_to_product)
)

Products indexed: 10000


In [ ]:


query_agent = QueryAgent()

print(
    "Query Agent ready."
)

Query Agent ready.


In [ ]:


def split_asins(
    value
):

    if pd.isna(value):
        return []

    return [
        item.strip()
        for item
        in str(value).split("|")
        if item.strip()
    ]


def get_recommended_asin(
    row
):

    possible_columns = [
        "recommended_asin",
        "recommended_product_asin",
        "final_asin"
    ]

    for column in possible_columns:

        if column in row.index:

            value = row.get(
                column
            )

            if (
                value is not None
                and not pd.isna(value)
                and str(value).strip()
                not in {
                    "",
                    "None",
                    "nan"
                }
            ):
                return str(
                    value
                ).strip()


    if (
        "top_5_asins"
        in row.index
    ):

        asins = split_asins(
            row[
                "top_5_asins"
            ]
        )

        if asins:
            return asins[0]

    return None

In [ ]:

def evaluate_product_quality(
    query,
    asin
):

    product = (
        asin_to_product.get(
            asin
        )
    )

    if product is None:

        return {
            "product_found": False,
            "product_type_match": False,
            "brand_match": False,
            "budget_requested": (
                query.budget
                is not None
            ),
            "budget_verifiable": False,
            "budget_match": None,
            "feature_status": "no_match",
            "feature_match_ratio": 0.0,
            "matched_features": [],
            "missing_features": (
                query.features
                or []
            ),
            "support_level": "unsupported"
        }


    title = str(
        product.get(
            "title",
            ""
        )
    )



    product_type_match = (
        check_product_type_match(
            query.product_type,
            title
        )
    )




    brand_match = (
        check_brand_match(
            query.brand,
            title
        )
    )


    # -------------------------------------------------
    # BUDGET
    # -------------------------------------------------

    budget_requested = (
        query.budget
        is not None
    )

    price = normalise_price(
        product.get(
            "price"
        )
    )

    budget_verifiable = (
        budget_requested
        and price is not None
    )

    budget_match = None

    if budget_verifiable:

        budget_match = (
            price
            <= float(
                query.budget
            )
        )


    # -------------------------------------------------
    # FEATURES
    # -------------------------------------------------

    (
        feature_status,
        feature_match_ratio,
        matched_features,
        missing_features

    ) = check_feature_match(

        query.features,
        product
    )


    # -------------------------------------------------
    # HARD CONSTRAINTS
    # -------------------------------------------------

    hard_constraints_ok = (
        product_type_match
        and brand_match
        and (
            budget_match
            is not False
        )
    )


    # -------------------------------------------------
    # SUPPORT LEVEL
    # -------------------------------------------------

    if not hard_constraints_ok:

        support_level = (
            "unsupported"
        )

    elif feature_status in {
        "full_match",
        "not_requested"
    }:

        support_level = (
            "grounded"
        )

    elif feature_status == (
        "partial_match"
    ):

        support_level = (
            "partial"
        )

    else:

        support_level = (
            "unsupported"
        )


    return {

        "product_found":
            True,

        "product_type_match":
            product_type_match,

        "brand_match":
            brand_match,

        "budget_requested":
            budget_requested,

        "budget_verifiable":
            budget_verifiable,

        "budget_match":
            budget_match,

        "feature_status":
            feature_status,

        "feature_match_ratio":
            feature_match_ratio,

        "matched_features":
            matched_features,

        "missing_features":
            missing_features,

        "support_level":
            support_level
    }

Evaluating top 5 products in each system

In [ ]:


system_dataframes = {

    "Single Agent":
        single_df,

    "Multi-Agent Without Coordination":
        no_coord_df,

    "Coordinated Multi-Agent":
        coordinated_df
}


quality_rows = []


for system_name, system_df in (
    system_dataframes.items()
):

    for _, row in (
        system_df.iterrows()
    ):

        test_id = int(
            row[
                "test_id"
            ]
        )

        user_query = str(
            row[
                "user_query"
            ]
        )

        query = (
            query_agent.parse(
                user_query
            )
        )

        returned_asins = (
            split_asins(
                row.get(
                    "top_5_asins",
                    ""
                )
            )
        )

        recommended_asin = (
            get_recommended_asin(
                row
            )
        )


        for rank, asin in enumerate(
            returned_asins,
            start=1
        ):

            quality = (
                evaluate_product_quality(
                    query,
                    asin
                )
            )

            quality_rows.append(
                {
                    "system":
                        system_name,

                    "test_id":
                        test_id,

                    "user_query":
                        user_query,

                    "rank":
                        rank,

                    "parent_asin":
                        asin,

                    "is_recommended":
                        asin
                        == recommended_asin,

                    "product_type_match":
                        quality[
                            "product_type_match"
                        ],

                    "brand_match":
                        quality[
                            "brand_match"
                        ],

                    "budget_requested":
                        quality[
                            "budget_requested"
                        ],

                    "budget_verifiable":
                        quality[
                            "budget_verifiable"
                        ],

                    "budget_match":
                        quality[
                            "budget_match"
                        ],

                    "feature_status":
                        quality[
                            "feature_status"
                        ],

                    "feature_match_ratio":
                        quality[
                            "feature_match_ratio"
                        ],

                    "matched_features":
                        " | ".join(
                            quality[
                                "matched_features"
                            ]
                        ),

                    "missing_features":
                        " | ".join(
                            quality[
                                "missing_features"
                            ]
                        ),

                    "support_level":
                        quality[
                            "support_level"
                        ]
                }
            )


quality_df = pd.DataFrame(
    quality_rows
)


display(
    quality_df.head(
        20
    )
)

,system,test_id,user_query,rank,parent_asin,is_recommended,product_type_match,brand_match,budget_requested,budget_verifiable,budget_match,feature_status,feature_match_ratio,matched_features,missing_features,support_level
0,Single Agent,1,Recommend a Samsung phone with a good camera a...,1,B09X9FC88M,True,True,True,False,False,None,full_match,1.0,camera | long battery life,,grounded
1,Single Agent,1,Recommend a Samsung phone with a good camera a...,2,B09S6VKCLX,False,True,True,False,False,None,full_match,1.0,camera | long battery life,,grounded
2,Single Agent,1,Recommend a Samsung phone with a good camera a...,3,B0BF1FZTLQ,False,True,True,False,False,None,full_match,1.0,camera | long battery life,,grounded
3,Single Agent,1,Recommend a Samsung phone with a good camera a...,4,B09WT8N5X7,False,True,True,False,False,None,full_match,1.0,camera | long battery life,,grounded
4,Single Agent,1,Recommend a Samsung phone with a good camera a...,5,B07QH32PCY,False,True,True,False,False,None,partial_match,0.5,camera,long battery life,partial
5,Single Agent,2,I want an Apple phone with excellent camera qu...,1,B078WZX9LD,True,True,True,False,False,None,full_match,1.0,camera,,grounded
6,Single Agent,2,I want an Apple phone with excellent camera qu...,2,B00CX0OZHY,False,True,True,False,False,None,full_match,1.0,camera,,grounded
7,Single Agent,2,I want an Apple phone with excellent camera qu...,3,B00BUYRQG6,False,True,True,False,False,None,full_match,1.0,camera,,grounded
8,Single Agent,3,Suggest a Google phone with fast performance.,1,B09NP4C1MF,True,True,True,False,False,None,no_match,0.0,,fast performance,unsupported
9,Single Agent,3,Suggest a Google phone with fast performance.,2,B08BXBT8MD,False,True,True,False,False,None,no_match,0.0,,fast performance,unsupported


Recommendation quality summary


In [ ]:


summary_rows = []


for system_name in (
    quality_df[
        "system"
    ].unique()
):

    system_quality = (
        quality_df[
            quality_df[
                "system"
            ]
            == system_name
        ]
    )

    brand_rows = (
        system_quality[
            system_quality[
                "brand_match"
            ].notna()
        ]
    )

    budget_rows = (
        system_quality[
            system_quality[
                "budget_verifiable"
            ]
            == True
        ]
    )


    total = len(
        system_quality
    )


    grounded_count = (
        (
            system_quality[
                "support_level"
            ]
            == "grounded"
        )
        .sum()
    )

    partial_count = (
        (
            system_quality[
                "support_level"
            ]
            == "partial"
        )
        .sum()
    )

    unsupported_count = (
        (
            system_quality[
                "support_level"
            ]
            == "unsupported"
        )
        .sum()
    )


    summary_rows.append(
        {
            "system":
                system_name,

            "products_evaluated":
                total,

            "product_type_accuracy":
                round(
                    system_quality[
                        "product_type_match"
                    ].mean(),
                    4
                ),

            "brand_match_rate":
                round(
                    system_quality[
                        "brand_match"
                    ].mean(),
                    4
                ),

            "budget_match_rate":
                round(
                    budget_rows[
                        "budget_match"
                    ].mean(),
                    4
                )
                if len(
                    budget_rows
                )
                else np.nan,

            "mean_feature_satisfaction":
                round(
                    system_quality[
                        "feature_match_ratio"
                    ].mean(),
                    4
                ),

            "grounded_product_rate":
                round(
                    grounded_count
                    / total,
                    4
                )
                if total
                else 0,

            "partial_support_rate":
                round(
                    partial_count
                    / total,
                    4
                )
                if total
                else 0,

            "unsupported_product_rate":
                round(
                    unsupported_count
                    / total,
                    4
                )
                if total
                else 0
        }
    )


quality_summary_df = pd.DataFrame(
    summary_rows
)


display(
    quality_summary_df
)

,system,products_evaluated,product_type_accuracy,brand_match_rate,budget_match_rate,mean_feature_satisfaction,grounded_product_rate,partial_support_rate,unsupported_product_rate
0,Single Agent,52,0.7885,1.0,1.0,0.7276,0.3846,0.2692,0.3462
1,Multi-Agent Without Coordination,59,1.0000,1.0,1.0,0.6045,0.4746,0.2542,0.2712
2,Coordinated Multi-Agent,45,1.0000,1.0,1.0,0.8370,0.6667,0.3333,0.0000


Final recommendation

In [ ]:


recommended_df = (
    quality_df[
        quality_df[
            "is_recommended"
        ]
        == True
    ]
    .copy()
)


final_quality_rows = []


for system_name in (
    system_dataframes.keys()
):

    system_results = (
        system_dataframes[
            system_name
        ]
    )

    total_queries = len(
        system_results
    )

    system_recommendations = (
        recommended_df[
            recommended_df[
                "system"
            ]
            == system_name
        ]
    )

    recommendation_count = len(
        system_recommendations
    )

    grounded = (
        (
            system_recommendations[
                "support_level"
            ]
            == "grounded"
        )
        .sum()
    )

    partial = (
        (
            system_recommendations[
                "support_level"
            ]
            == "partial"
        )
        .sum()
    )

    unsupported = (
        (
            system_recommendations[
                "support_level"
            ]
            == "unsupported"
        )
        .sum()
    )


    final_quality_rows.append(
        {
            "system":
                system_name,

            "total_queries":
                total_queries,

            "recommendations_made":
                recommendation_count,

            "recommendation_rate":
                round(
                    recommendation_count
                    / total_queries,
                    4
                ),

            "grounded_recommendation_rate":
                round(
                    grounded
                    / recommendation_count,
                    4
                )
                if recommendation_count
                else 0,

            "partial_grounding_rate":
                round(
                    partial
                    / recommendation_count,
                    4
                )
                if recommendation_count
                else 0,

            "hallucination_rate":
                round(
                    unsupported
                    / recommendation_count,
                    4
                )
                if recommendation_count
                else 0
        }
    )


final_quality_df = (
    pd.DataFrame(
        final_quality_rows
    )
)


display(
    final_quality_df
)

,system,total_queries,recommendations_made,recommendation_rate,grounded_recommendation_rate,partial_grounding_rate,hallucination_rate
0,Single Agent,15,13,0.8667,0.6923,0.1538,0.1538
1,Multi-Agent Without Coordination,15,15,1.0000,0.4667,0.2000,0.3333
2,Coordinated Multi-Agent,15,12,0.8000,0.7500,0.2500,0.0000


saving everything

In [ ]:

quality_df.to_csv(
    RESULTS_FOLDER
    / "recommendation_quality_details.csv",
    index=False
)


quality_summary_df.to_csv(
    RESULTS_FOLDER
    / "recommendation_quality_summary.csv",
    index=False
)


final_quality_df.to_csv(
    RESULTS_FOLDER
    / "final_recommendation_grounding_summary.csv",
    index=False
)


print(
    "Recommendation-quality evaluation saved."
)

Recommendation-quality evaluation saved.
